# 08 — Registry, alias, and inference contract

**Objectives**

- Register only an adopted exact artifact.
- Attach decision evidence and move the `champion` alias conditionally.
- Reload by alias and apply the versioned threshold to representative inputs.

**Prerequisite:** lesson 07 produced a decision.


In [ ]:
import pandas as pd

from aai_local_classification.contracts import SplitName
from aai_local_classification.data import load_split
from aai_local_classification.inference import load_champion
from aai_local_classification.tracking import local_paths
from aai_local_classification.workflow import (
    ensure_prepared,
    get_or_run_candidate_selection,
    promote_if_approved,
    run_frozen_test_gate,
)
from aai_local_classification.learning import study_root
from aai_local_classification.settings import load_settings

settings = load_settings()
root = study_root()
print(f"Course state: {root}")
print(f"Experiment: {settings.experiment_name}")


In [ ]:
selected = get_or_run_candidate_selection(settings, root)
decision = run_frozen_test_gate(settings, root, selected)
promotion = promote_if_approved(settings, decision, root, selected)
promotion


In [ ]:
if not promotion["registered"]:
    raise RuntimeError(
        "The learning model did not pass; inspect lesson 07 instead of forcing registration."
    )
predictor = load_champion(settings, root)
paths = local_paths(root)
ensure_prepared(settings, root)
validation = load_split(settings, SplitName.VALIDATION, paths.data_root)
predictions = predictor.predict(validation.head(8), settings)
pd.concat([validation[["account_id"]].head(8), predictions], axis=1)


In [ ]:
assert predictor.model_version == str(promotion["model_version"])
assert predictor.threshold == decision.threshold
assert predictions["churn_probability"].between(0, 1).all()
print("Alias resolution, concrete version, threshold, and output schema agree.")


`champion` is a mutable pointer, while the resolved version is concrete evidence.
Record that version in every batch/deployment. The threshold is stored both in
the logged model metadata and the registered version tags; the inference helper
uses the approved version tag and tests parity with decision evidence.

### Exercise

Why should an online endpoint update to a concrete resolved version instead of
assuming that moving an alias changes a running deployment?

**Hint:** discovery/promotion state and deployment configuration have different
lifecycles, permissions, and rollback behavior.

**Checkpoint:** registration is conditional, the model has a signature and input
example, and a fresh loader reproduces bounded probabilities and binary actions.
Promotion also rechecks the dataset, selected run/model ID, policy digests, and
threshold before moving the alias.

Next: **09_monitoring_and_databricks.ipynb**.
